# ✈️ YOLOv8 Aircraft Surface Damage — Colab Training
> ⚠️ **First:** Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Step 1 — Install dependencies
!pip install ultralytics -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.5 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
# Step 2 — Upload your dataset ZIP
# ZIP your dataset folder (containing train/, valid/, test/ folders) and upload it here
from google.colab import files
import zipfile, os

print('Select your dataset ZIP file to upload...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'Extracting {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')

print('Files extracted:')
!ls /content/dataset

Select your dataset ZIP file to upload...


Saving corrosion-seg.v2i.yolov8.zip to corrosion-seg.v2i.yolov8.zip
Extracting corrosion-seg.v2i.yolov8.zip...
Files extracted:
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [ ]:
# Step 3 — Create data.yaml
import yaml

data_config = {
    'path': '/content/dataset',
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 3,
    'names': ['corrosion', 'crack', 'dent']
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print('data.yaml created:')
!cat /content/data.yaml

data.yaml created:
names:
- corrosion
- crack
- dent
nc: 3
path: /content/dataset
test: test/images
train: train/images
val: valid/images


In [ ]:
# Step 4 — Train on 100% dataset
from ultralytics import YOLO
import torch

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    fraction=1.0,      # 100% of dataset
    batch=16,          # lower to 8 if out of memory
    workers=4,
    device='cuda',
    patience=20,
    project='/content/runs',
    name='train',
    save=True,
    plots=True
)

print('\n✅ Training complete!')
print(f'Best weights: /content/runs/train/weights/best.pt')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0

KeyboardInterrupt: 

In [ ]:
# Step 5 — Validate on test set
metrics = model.val(data='/content/data.yaml', split='test')

print('\n📊 Results:')
print(f'  mAP@50:    {metrics.box.map50:.4f}')
print(f'  mAP@50-95: {metrics.box.map:.4f}')
print(f'  Precision: {metrics.box.mp:.4f}')
print(f'  Recall:    {metrics.box.mr:.4f}')

Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 13,065 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1006.5±475.5 MB/s, size: 18.6 KB)
val: Scanning /content/dataset/test/labels... 148 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 148/148 1.8Kit/s 0.1s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 3.4it/s 3.0s
                   all        148        627      0.657      0.426      0.434      0.236
             corrosion         54        369      0.662      0.542      0.582      0.375
                 crack         62        150      0.626       0.44      0.421      0.181
                  dent         36        108      0.684      0.296      0.297      0.153
Speed: 1.7ms preprocess, 5.3ms inference, 0.0ms loss, 4.8ms postprocess per image


In [ ]:
# Step 6 — Download best.pt
from google.colab import files
import shutil, os

best_pt = '/content/runs/train/weights/best.pt'

if os.path.exists(best_pt):
    shutil.copy(best_pt, '/content/best.pt')
    files.download('/content/best.pt')
    print('✅ Downloading best.pt...')
else:
    print('❌ best.pt not found. Check training output above.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloading best.pt...
